In [1]:
!pip install numpy
!pip install pandas
!pip install sklearn
!pip install tensorflow

  error: subprocess-exited-with-error
  
  × python setup.py egg_info did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Preparing metadata (setup.py) ... error
error: metadata-generation-failed

× Encountered error while generating package metadata.
╰─> See above for output.

note: This is an issue with the package mentioned above, not pip.
hint: See above for details.


In [2]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error,r2_score
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout

In [29]:
df= pd.read_csv("/content/cleaned_pump_data.csv")
df["date"] = pd.to_datetime(df["date"])
df=df.drop(columns=["ID"])


In [30]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 26 columns):
 #   Column                                Non-Null Count  Dtype         
---  ------                                --------------  -----         
 0   date                                  1000 non-null   datetime64[ns]
 1   pressure                              1000 non-null   float64       
 2   flow_rate                             1000 non-null   float64       
 3   suction_pressure                      1000 non-null   float64       
 4   discharge_pressure                    1000 non-null   float64       
 5   rpm                                   1000 non-null   float64       
 6   vibration_rms                         1000 non-null   float64       
 7   vibration_peak                        1000 non-null   float64       
 8   structural_vibration                  1000 non-null   float64       
 9   motor_current                         1000 non-null   float64       
 10  p

In [31]:
data=df[["time_until_next_maintenance"]]

In [32]:
scaler = MinMaxScaler(feature_range=(0, 1))
scaled_data = scaler.fit_transform(data)

In [33]:
def create_sequences(data, lookback):
    X, y = [], []
    for i in range(lookback, len(data)):
        X.append(data[i - lookback:i, 0])
        y.append(data[i, 0])
    return np.array(X), np.array(y)

In [34]:
lookback = 30
X, y = create_sequences(scaled_data, lookback)

# Reshape for LSTM: (samples, timesteps, features)
X = X.reshape((X.shape[0], X.shape[1], 1))

In [35]:
X = X.reshape((X.shape[0], X.shape[1], 1))
split_index = int(len(X) * 0.8)

X_train, X_test = X[:split_index], X[split_index:]
y_train, y_test = y[:split_index], y[split_index:]



In [36]:
model = Sequential([
    LSTM(64, return_sequences=True, input_shape=(X_train.shape[1], 1)),
    Dropout(0.2),
    LSTM(32),
    Dropout(0.2),
    Dense(1)
])

model.compile(optimizer="adam", loss="mse")

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [37]:
history = model.fit(
    X_train, y_train,
    epochs=30,
    batch_size=32,
    validation_data=(X_test, y_test),
    verbose=1
)

Epoch 1/30
23/23 ━━━━━━━━━━━━━━━━━━━━ 6s 88ms/step - loss: 0.0925 - val_loss: 0.0357
Epoch 2/30
23/23 ━━━━━━━━━━━━━━━━━━━━ 2s 69ms/step - loss: 0.0431 - val_loss: 0.0296
Epoch 3/30
23/23 ━━━━━━━━━━━━━━━━━━━━ 2s 69ms/step - loss: 0.0363 - val_loss: 0.0267
Epoch 4/30
23/23 ━━━━━━━━━━━━━━━━━━━━ 3s 69ms/step - loss: 0.0310 - val_loss: 0.0245
Epoch 5/30
23/23 ━━━━━━━━━━━━━━━━━━━━ 3s 113ms/step - loss: 0.0312 - val_loss: 0.0230
Epoch 6/30
23/23 ━━━━━━━━━━━━━━━━━━━━ 2s 76ms/step - loss: 0.0288 - val_loss: 0.0217
Epoch 7/30
23/23 ━━━━━━━━━━━━━━━━━━━━ 2s 70ms/step - loss: 0.0256 - val_loss: 0.0210
Epoch 8/30
23/23 ━━━━━━━━━━━━━━━━━━━━ 2s 69ms/step - loss: 0.0264 - val_loss: 0.0192
Epoch 9/30
23/23 ━━━━━━━━━━━━━━━━━━━━ 2s 69ms/step - loss: 0.0248 - val_loss: 0.0185
Epoch 10/30
23/23 ━━━━━━━━━━━━━━━━━━━━ 3s 73ms/step - loss: 0.0217 - val_loss: 0.0167
Epoch 11/30
23/23 ━━━━━━━━━━━━━━━━━━━━ 2s 79ms/step - loss: 0.0217 - val_loss: 0.0162
Epoch 12/30
23/23 ━━━━━━━━━━━━━━━━━━━━ 3s 109ms/step - loss: 0

In [38]:
y_pred = model.predict(X_test)

6/6 ━━━━━━━━━━━━━━━━━━━━ 1s 74ms/step


In [39]:
y_test_actual = scaler.inverse_transform(y_test.reshape(-1, 1))
y_pred_actual = scaler.inverse_transform(y_pred)

In [40]:
rmse = np.sqrt(mean_squared_error(y_test_actual, y_pred_actual))
mae = mean_absolute_error(y_test_actual, y_pred_actual)
y_mean = np.mean(y_test_actual)

ss_total = np.sum((y_test_actual - y_mean) ** 2)
ss_residual = np.sum((y_test_actual - y_pred_actual) ** 2)

r2 = 1 - (ss_residual / ss_total)

print("R²:", r2)
print("RMSE:", rmse)
print("MAE:", mae)

R²: 0.7507308767255776
RMSE: 7.3344971853132055
MAE: 3.534361100458837
